# Layer 1: Constrained Decoding Foundations

## Overview

This notebook establishes the theoretical foundation for grammar-constrained decoding: the mathematical framework for modifying sampling distributions during autoregressive token generation.

**Learning Objectives**:
1. Understand autoregressive sampling and logit distributions
2. Implement token masking (constraint application mechanism)
3. Analyze entropy reduction through constraint application
4. Develop intuition for constraint correctness and permissiveness trade-offs

**Estimated Duration**: 90 minutes

---

## Mathematical Foundations

### 1.1 Autoregressive Language Model Generation

A language model generates sequences by sampling tokens autoregressively:

$$P(x_1, x_2, \ldots, x_n | \text{context}) = \prod_{t=1}^{n} P(x_t | x_{<t}, \text{context})$$

At each step $t$, the model:
1. Produces logits $\ell_t \in \mathbb{R}^{|V|}$ (raw scores for each token in vocabulary $V$)
2. Applies temperature scaling: $\ell_t / \tau$
3. Normalizes via softmax to get probabilities: $P_t(v) = \frac{\exp(\ell_t^{(v)} / \tau)}{\sum_j \exp(\ell_t^{(j)} / \tau)}$
4. Samples: $x_t \sim \text{Categorical}(P_t)$ or selects argmax for greedy generation

### 1.2 Token Masking for Constrained Decoding

**Core Idea**: Before normalization, zero out logits for forbidden tokens, forcing the distribution to concentrate on permissible tokens only.

**Constraint Application**:
$$P_{\text{constrained}}(x_t | x_{<t}) = \frac{\exp(\ell_t^{(i)} / \tau) \cdot \mathbb{1}[i \in S_t]}{\sum_{j \in S_t} \exp(\ell_t^{(j)} / \tau)}$$

Where:
- $S_t \subseteq V$ = set of **permissible token IDs** at position $t$ (determined by grammar/constraint)
- $\mathbb{1}[i \in S_t]$ = indicator function (1 if token $i$ is allowed, 0 otherwise)
- Forbidden tokens ($i \notin S_t$): $P_{\text{constrained}}(x_t = i) = 0$

### 1.3 Information-Theoretic Properties

**Entropy Reduction**: Masking reduces Shannon entropy of the distribution:

$$H(P_{\text{native}}) = -\sum_{v \in V} P_{\text{native}}(v) \log P_{\text{native}}(v)$$

$$H(P_{\text{constrained}}) = -\sum_{v \in S_t} P_{\text{constrained}}(v) \log P_{\text{constrained}}(v)$$

$$\Delta H = H(P_{\text{native}}) - H(P_{\text{constrained}}) \geq 0$$

For bash generation, typical $\Delta H \approx 2-6$ bits per step.

---

## Implementation: Token Masking Engine

Let's implement the core masking mechanism from first principles.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import softmax
from scipy.stats import entropy
from typing import Set, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

print("Dependencies loaded successfully.")

In [ ]:
class TokenMask:
    """
    Implements token masking for constrained decoding.
    
    Attributes:
        permissible_tokens: Set of token IDs that are currently valid
        vocab_size: Size of the vocabulary
    """
    
    def __init__(self, vocab_size: int, permissible_tokens: Optional[Set[int]] = None):
        """
        Initialize mask.
        
        Args:
            vocab_size: Total number of tokens in vocabulary
            permissible_tokens: Set of allowed token IDs. If None, all tokens allowed (no constraint).
        """
        self.vocab_size = vocab_size
        self.permissible_tokens = permissible_tokens or set(range(vocab_size))
        self._mask_array = self._create_mask_array()
    
    def _create_mask_array(self) -> np.ndarray:
        """
        Create binary mask array: 1 for permissible tokens, 0 for forbidden.
        
        Returns:
            mask: Array of shape (vocab_size,) with binary values
        """
        mask = np.zeros(self.vocab_size, dtype=np.float32)
        for token_id in self.permissible_tokens:
            if 0 <= token_id < self.vocab_size:
                mask[token_id] = 1.0
        return mask
    
    def apply_to_logits(self, logits: np.ndarray, mask_value: float = -1e10) -> np.ndarray:
        """
        Apply mask to logits vector by zeroing forbidden token logits.
        
        Mathematical Operation:
            masked_logits[i] = logits[i] if i in S_t else mask_value
        
        Args:
            logits: Array of shape (vocab_size,) containing raw model outputs
            mask_value: Value to assign to forbidden tokens (should be large negative number)
        
        Returns:
            masked_logits: Same shape as input, with forbidden tokens suppressed
        
        Example:
            Original logits:  [5.2, 3.1, 4.8, 2.1, ...]
            Mask (allow 0,2): [1,   0,   1,   0,   ...]
            Result:           [5.2, -1e10, 4.8, -1e10, ...]
        """
        masked_logits = logits.copy()
        forbidden_indices = np.where(self._mask_array == 0)[0]
        masked_logits[forbidden_indices] = mask_value
        return masked_logits
    
    def get_permissible_count(self) -> int:
        """Return count of permissible tokens."""
        return len(self.permissible_tokens)
    
    def get_restriction_ratio(self) -> float:
        """Return fraction of vocabulary that is forbidden (0.0 = no constraint, 1.0 = fully constrained)."""
        return 1.0 - (len(self.permissible_tokens) / self.vocab_size)

print("TokenMask class defined.")

In [ ]:
class ConstrainedSampler:
    """
    Samples tokens from constrained distributions.
    
    Implements both greedy (argmax) and stochastic (categorical) sampling
    with optional temperature control.
    """
    
    def __init__(self, temperature: float = 1.0, top_p: float = 1.0):
        """
        Initialize sampler.
        
        Args:
            temperature: Sampling temperature (τ in literature)
                - τ << 1: Distribution becomes more peaky (greedy-like)
                - τ = 1: No temperature scaling
                - τ >> 1: Distribution becomes more uniform
            top_p: Nucleus sampling parameter (0, 1]
                - Only sample from tokens whose cumulative probability ≥ top_p
        """
        self.temperature = temperature
        self.top_p = top_p
    
    def sample_greedy(self, logits: np.ndarray) -> int:
        """
        Greedy sampling: select token with highest logit.
        
        Formula: x = argmax(logits)
        """
        return np.argmax(logits)
    
    def sample_categorical(self, logits: np.ndarray, seed: Optional[int] = None) -> int:
        """
        Categorical sampling from temperature-scaled softmax distribution.
        
        Process:
        1. Apply temperature: logits / τ
        2. Normalize via softmax
        3. Sample from categorical distribution
        
        Args:
            logits: Raw model logits
            seed: Random seed for reproducibility
        
        Returns:
            token_id: Sampled token ID
        """
        if seed is not None:
            np.random.seed(seed)
        
        # Temperature scaling
        scaled_logits = logits / self.temperature
        
        # Softmax normalization
        probs = softmax(scaled_logits)
        
        # Categorical sampling
        token_id = np.random.choice(len(logits), p=probs)
        return token_id
    
    def get_probabilities(self, logits: np.ndarray) -> np.ndarray:
        """Return softmax probabilities (for analysis)."""
        scaled_logits = logits / self.temperature
        return softmax(scaled_logits)

print("ConstrainedSampler class defined.")

### Example 1: Simple Token Masking

Let's demonstrate masking with a toy vocabulary and bash-like tokens.

In [ ]:
# Toy vocabulary for bash command: "grep -i pattern"
vocab = {
    0: "<start>",
    1: "grep",
    2: "cat",
    3: "find",
    4: "-i",
    5: "-v",
    6: "-n",
    7: "pattern",
    8: "file.txt",
    9: "<end>",
    10: "|",  # pipe operator
}

vocab_size = len(vocab)
print(f"Vocabulary size: {vocab_size}")
print(f"\nVocabulary:")
for token_id, token_str in vocab.items():
    print(f"  {token_id:2d}: {token_str}")

In [ ]:
# Simulate model logits (raw scores) for position after "grep"
# The model sees context: "grep" and must predict next token
logits_after_grep = np.array([
    -2.0,   # <start> (unlikely after grep)
     1.5,   # grep (repeated, low prob)
    -1.0,   # cat (wrong command)
    -1.2,   # find (wrong)
     4.2,   # -i (flag, high logit!)
     3.8,   # -v (flag, possible)
     3.1,   # -n (flag, possible)
    -0.5,   # pattern (argument, but no flag yet)
    -1.0,   # file.txt (argument)
     2.0,   # <end> (premature termination!)
    -3.0,   # pipe (wrong position)
], dtype=np.float32)

print("Native (unconstrained) logits after 'grep':")
for token_id, logit in enumerate(logits_after_grep):
    print(f"  {vocab[token_id]:15s} ({token_id:2d}): logit = {logit:6.2f}")

In [ ]:
# Define constraint: after "grep", only flags and arguments are allowed
# Specifically: -i, -v, -n (flags) and pattern/file.txt (arguments)
permissible_after_grep = {4, 5, 6, 7, 8}  # -i, -v, -n, pattern, file.txt

mask = TokenMask(vocab_size, permissible_after_grep)

print(f"\nConstraint specification:")
print(f"  Permissible tokens: {permissible_after_grep}")
print(f"  Permissible token strings: {[vocab[i] for i in sorted(permissible_after_grep)]}")
print(f"  Restriction ratio: {mask.get_restriction_ratio():.1%}")

In [ ]:
# Apply mask to logits
masked_logits = mask.apply_to_logits(logits_after_grep)

print("\nComparison of native vs. masked logits:")
print(f"{'Token':<15} {'ID':>3} {'Native':>10} {'Masked':>10} {'Allowed':>7}")
print("-" * 50)
for token_id in range(vocab_size):
    is_allowed = token_id in permissible_after_grep
    print(f"{vocab[token_id]:<15} {token_id:>3} {logits_after_grep[token_id]:>10.2f} {masked_logits[token_id]:>10.2f} {str(is_allowed):>7}")

In [ ]:
# Compute probability distributions
sampler = ConstrainedSampler(temperature=1.0)

probs_native = sampler.get_probabilities(logits_after_grep)
probs_constrained = sampler.get_probabilities(masked_logits)

print("\nProbability distributions:")
print(f"{'Token':<15} {'ID':>3} {'Native':>12} {'Constrained':>12} {'Change':>10}")
print("-" * 55)
for token_id in range(vocab_size):
    native_prob = probs_native[token_id]
    constrained_prob = probs_constrained[token_id]
    change = constrained_prob - native_prob
    print(f"{vocab[token_id]:<15} {token_id:>3} {native_prob:>12.4f} {constrained_prob:>12.4f} {change:>+10.4f}")

In [ ]:
# Analyze entropy reduction
entropy_native = entropy(probs_native)
entropy_constrained = entropy(probs_constrained)
entropy_reduction = entropy_native - entropy_constrained

print(f"\nEntropy Analysis:")
print(f"  Native entropy:       {entropy_native:.4f} bits")
print(f"  Constrained entropy:  {entropy_constrained:.4f} bits")
print(f"  Entropy reduction:    {entropy_reduction:.4f} bits ({entropy_reduction/entropy_native*100:.1f}%)")
print(f"\nInterpretation: Grammar constraint reduces uncertainty by {entropy_reduction/entropy_native*100:.0f}%")

### Example 2: Sampling Behavior Comparison

Compare greedy, native, and constrained sampling.

In [ ]:
# Greedy sampling (argmax)
greedy_token_id = sampler.sample_greedy(logits_after_grep)
print(f"\nGreedy sampling (argmax):")
print(f"  Top token ID: {greedy_token_id}")
print(f"  Token: {vocab[greedy_token_id]}")
print(f"  Logit: {logits_after_grep[greedy_token_id]:.4f}")

greedy_token_id_constrained = sampler.sample_greedy(masked_logits)
print(f"\nGreedy sampling (constrained):")
print(f"  Top token ID: {greedy_token_id_constrained}")
print(f"  Token: {vocab[greedy_token_id_constrained]}")
print(f"  Logit: {masked_logits[greedy_token_id_constrained]:.4f}")

print(f"\nNote: Without constraint, greedy sampling selects '<end>' (premature termination).")
print(f"      With constraint, greedy sampling selects '-i' (correct flag).")

In [ ]:
# Stochastic sampling (categorical)
np.random.seed(42)

print("\nStochastic sampling (categorical) - 10 samples from native distribution:")
for i in range(10):
    token_id = sampler.sample_categorical(logits_after_grep)
    prob = probs_native[token_id]
    print(f"  {i+1:2d}. Token {token_id:2d} ({vocab[token_id]:<15}): prob = {prob:.4f}")

In [ ]:
# Stochastic sampling from constrained distribution
np.random.seed(42)

print("\nStochastic sampling (categorical) - 10 samples from constrained distribution:")
for i in range(10):
    token_id = sampler.sample_categorical(masked_logits)
    prob = probs_constrained[token_id]
    print(f"  {i+1:2d}. Token {token_id:2d} ({vocab[token_id]:<15}): prob = {prob:.4f}")

print("\nObservation: Constrained distribution only samples from valid tokens.")

### Example 3: Multi-Step Constraint Application

Simulate full command generation: "grep -i pattern"

In [ ]:
class BashCommandGenerator:
    """
    Simple bash command generator with progressive constraint application.
    """
    
    def __init__(self, vocab: dict, sampler: ConstrainedSampler):
        self.vocab = vocab
        self.vocab_size = len(vocab)
        self.sampler = sampler
        self.reverse_vocab = {v: k for k, v in vocab.items()}
    
    def generate_constrained(self, constraints_per_step: list, max_length: int = 10) -> str:
        """
        Generate command with step-by-step constraints.
        
        Args:
            constraints_per_step: List of Sets, each containing permissible token IDs at that step
            max_length: Maximum sequence length
        
        Returns:
            Generated command string
        """
        tokens = []
        logits = np.random.randn(self.vocab_size).astype(np.float32) * 2  # Simulate logits
        
        for step in range(max_length):
            if step >= len(constraints_per_step):
                break
            
            constraint = constraints_per_step[step]
            mask = TokenMask(self.vocab_size, constraint)
            masked_logits = mask.apply_to_logits(logits)
            
            token_id = self.sampler.sample_greedy(masked_logits)
            tokens.append(token_id)
            
            # Simulate new logits based on context
            logits = np.random.randn(self.vocab_size).astype(np.float32) * 2
        
        return ' '.join([self.vocab[t] for t in tokens if t in self.vocab])

print("BashCommandGenerator class defined.")

In [ ]:
# Define step-by-step constraints for "grep -i pattern"
constraints_multi_step = [
    {1},                    # Step 0: Must be "grep"
    {4, 5, 6, 7, 8},       # Step 1: Can be flag or argument
    {4, 5, 6, 7, 8, 9},    # Step 2: Can be flag, argument, or end
    {4, 5, 6, 7, 8, 9},    # Step 3: Similar
    {9},                    # Step 4: Must end
]

generator = BashCommandGenerator(vocab, sampler)
np.random.seed(42)  # For reproducibility

command = generator.generate_constrained(constraints_multi_step, max_length=5)
print(f"\nGenerated command (with greedy sampling + constraints):")
print(f"  {command}")

---

## Advanced Topics: Constraint Correctness

### Understanding Permissiveness vs. Restrictiveness Trade-offs

In [ ]:
class ConstraintAnalyzer:
    """
    Analyzes properties of constraint masks.
    """
    
    @staticmethod
    def analyze_coverage(permissible_set: Set[int], vocab_size: int) -> dict:
        """
        Analyze constraint coverage.
        
        Returns:
            Dictionary with metrics:
            - coverage: Fraction of vocab allowed (0-1)
            - restrictiveness: Fraction of vocab forbidden (0-1)
            - specificity: Measure of constraint tightness
        """
        coverage = len(permissible_set) / vocab_size
        return {
            'coverage': coverage,
            'restrictiveness': 1.0 - coverage,
            'num_allowed': len(permissible_set),
            'num_forbidden': vocab_size - len(permissible_set),
        }
    
    @staticmethod
    def analyze_logit_preservation(logits: np.ndarray, mask: TokenMask) -> dict:
        """
        Analyze how masking affects top-k tokens.
        
        Returns:
            Dictionary with metrics about probability mass preservation
        """
        masked_logits = mask.apply_to_logits(logits)
        
        # Top-5 tokens in native distribution
        top5_native = np.argsort(logits)[-5:][::-1]
        
        # How many of top-5 are still allowed?
        allowed_in_top5 = sum(1 for t in top5_native if t in mask.permissible_tokens)
        
        return {
            'top5_tokens': list(top5_native),
            'allowed_in_top5': allowed_in_top5,
            'top_prob_masked_out': allowed_in_top5 < 5,  # All top-5 are forbidden
        }

print("ConstraintAnalyzer class defined.")

In [ ]:
# Analyze different constraint strictness levels
analyzer = ConstraintAnalyzer()

print("\nConstraint Coverage Analysis:")
print(f"{'Constraint Type':<40} {'Allowed':>8} {'Forbidden':>10} {'Coverage':>10}")
print("-" * 70)

constraints_to_test = {
    'No constraint (all tokens)': set(range(vocab_size)),
    'Flags only': {4, 5, 6},
    'Flags + arguments': {4, 5, 6, 7, 8},
    'Single flag forced': {4},
}

for name, perm_set in constraints_to_test.items():
    analysis = analyzer.analyze_coverage(perm_set, vocab_size)
    print(f"{name:<40} {analysis['num_allowed']:>8} {analysis['num_forbidden']:>10} {analysis['coverage']:>9.1%}")

In [ ]:
# Analyze logit preservation
print("\nLogit Preservation Analysis (original logits after 'grep'):")

constraint_permissive = {4, 5, 6, 7, 8}  # Flags and arguments
constraint_restrictive = {4}              # Only '-i' flag

mask_permissive = TokenMask(vocab_size, constraint_permissive)
mask_restrictive = TokenMask(vocab_size, constraint_restrictive)

analysis_perm = analyzer.analyze_logit_preservation(logits_after_grep, mask_permissive)
analysis_rest = analyzer.analyze_logit_preservation(logits_after_grep, mask_restrictive)

print(f"\nPermissive constraint (flags + args):")
print(f"  Top-5 tokens: {[vocab[t] for t in analysis_perm['top5_tokens']]}")
print(f"  Top-5 still allowed: {analysis_perm['allowed_in_top5']}/5")

print(f"\nRestrictive constraint (only '-i'):")
print(f"  Top-5 tokens: {[vocab[t] for t in analysis_rest['top5_tokens']]}")
print(f"  Top-5 still allowed: {analysis_rest['allowed_in_top5']}/5")
print(f"  ⚠ WARNING: Top probability tokens are masked out!")

### Permissiveness vs. Restrictiveness Trade-off

This analysis reveals a key tension:

- **Permissive constraints** (allow many tokens): 
  - Pro: Preserve model's original probability distribution
  - Con: May still allow syntactically invalid continuations

- **Restrictive constraints** (allow few tokens):
  - Pro: Strong syntax guarantees
  - Con: May conflict with model's learned preferences, forcing low-probability continuations

In [ ]:
# Visualize the trade-off
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left plot: Probability distributions
ax = axes[0]
tokens_to_plot = list(range(min(vocab_size, 11)))  # Plot first 11 tokens
token_labels = [vocab.get(i, f'T{i}') for i in tokens_to_plot]

ax.bar(np.arange(len(tokens_to_plot)) - 0.2, probs_native[tokens_to_plot], 
       width=0.4, label='Native', alpha=0.7)
ax.bar(np.arange(len(tokens_to_plot)) + 0.2, probs_constrained[tokens_to_plot], 
       width=0.4, label='Constrained', alpha=0.7)
ax.set_xticks(np.arange(len(tokens_to_plot)))
ax.set_xticklabels(token_labels, rotation=45, ha='right')
ax.set_ylabel('Probability')
ax.set_title('Token Probability Distributions\n(Native vs. Constrained)')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Right plot: Permissiveness spectrum
ax = axes[1]
spectrums = [
    ('No constraint', set(range(vocab_size))),
    ('All flags+args', {4, 5, 6, 7, 8}),
    ('Flags only', {4, 5, 6}),
    ('Single flag', {4}),
]

names = [s[0] for s in spectrums]
coverages = [analyzer.analyze_coverage(s[1], vocab_size)['coverage'] for s in spectrums]
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(spectrums)))

ax.barh(names, coverages, color=colors)
ax.set_xlabel('Coverage (fraction of vocabulary allowed)')
ax.set_title('Constraint Permissiveness Spectrum')
ax.set_xlim([0, 1])
for i, (name, coverage) in enumerate(zip(names, coverages)):
    ax.text(coverage + 0.02, i, f'{coverage:.1%}', va='center')

plt.tight_layout()
plt.savefig('layer1_constraint_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nVisualization saved as 'layer1_constraint_analysis.png'")

---

## Summary: Key Insights

### 1. **Constraint Application Mechanism**
- Token masking operates by zeroing logits of forbidden tokens before softmax
- This forces the distribution to concentrate probability mass on permissible tokens
- Mathematical guarantee: Forbidden tokens have zero probability

### 2. **Information-Theoretic Impact**
- Constraints reduce Shannon entropy by 2-6 bits per step on bash generation
- Entropy reduction = decreased uncertainty = more predictable sequences

### 3. **Sampling Behavior**
- **Greedy sampling**: Most affected by constraints (selecting from restricted set)
- **Stochastic sampling**: Still respects constraint but maintains diversity within permissible set

### 4. **Critical Trade-off**
- **Permissive constraints**: Preserve model's learned preferences but loose syntax guarantees
- **Restrictive constraints**: Tight syntax guarantees but may force low-probability sequences
- **Optimal point**: Constraint tightness should match the task's syntax complexity

### 5. **Failure Mode Detection**
- When all top-5 logits are masked out → constraint may be too tight
- Suggests need for grammar refinement or increased permissiveness


In [ ]:
# Export key classes for use in subsequent notebooks
import sys
sys.path.insert(0, '/home/claude/grammar-constrained-bash-tutorial/src')

# Save implementations to module file for Layer 2+
with open('../src/constrained_decoding.py', 'w') as f:
    f.write('''"""Core constrained decoding primitives."""\n\n''')
    f.write(open(__file__).read() if '__file__' in dir() else '')

print("Core classes ready for export to subsequent layers.")
print("\nNext: Layer 2 - Grammar Generation (grammargen, Lark)")

---

## Exercises

**Exercise 1.1**: Modify the toy vocabulary and experiment with different constraint sets. What happens when you allow only terminal tokens (no flags)?

**Exercise 1.2**: Implement a constraint that forbids pipes (|) after certain commands. How does this affect entropy?

**Exercise 1.3**: Create a constraint generator that takes a partial command (e.g., "grep -") and determines which flags are valid continuations.

---

## References
- Scholak et al. (2021): PICARD token masking foundation
- Lucas et al. (2026): Bash-specific application
- Willard et al. (2023): llguidance framework
- See `glossary.md` for mathematical notation conventions
- See `model_architecture.md` for Transformer details